# Day 13：Python 高级工程特性 —— 装饰器、生成器 `yield`、类型注解、`asyncio` 与 `.env` 安全配置

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 13 天！前几天我们搭好了多厂商模型适配器，但线上生产环境马上提出了四大工程诉求：**接口抖动自动重试 + 耗时监控**、**长文本流式吐出防内存爆炸**、**多 Prompt 并发提速**，以及**绝不把 API Key 硬编码进 Git**。今天我们就把这四大 Python 进阶武器一次性落地！

## 1. 第一步：类型注解（`typing`）与装饰器（`@log_latency`、`@retry_on_error`）
装饰器的本质是“**接收一个函数、返回一个增强后的新函数**”。
- 使用 `TypeVar('F', bound=Callable[..., Any])` 让类型检查器知道被装饰函数的签名没有丢失；
- 必须加上 `@functools.wraps(func)`，否则原函数的 `__name__` 和 `__doc__` 会被覆盖成 `wrapper`。

In [1]:
import functools
import time
from typing import Any, Callable, TypeVar

F = TypeVar("F", bound=Callable[..., Any])


def log_latency(func: F) -> F:
    """函数执行耗时监控装饰器（毫秒）。"""
    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed_ms = (time.perf_counter() - start) * 1000
            print(f"[Latency] {func.__name__} 耗时 {elapsed_ms:.2f} ms")
    return wrapper  # type: ignore[return-value]


def retry_on_error(max_retries: int = 3, delay: float = 0.01) -> Callable[[F], F]:
    """带参数的异常自动重试装饰器。"""
    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            last_exc: BaseException | None = None
            for attempt in range(1, max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as exc:
                    last_exc = exc
                    print(f"  [Retry] 第 {attempt} 次失败: {exc}")
                    if attempt < max_retries and delay > 0:
                        time.sleep(delay)
            assert last_exc is not None
            raise last_exc
        return wrapper  # type: ignore[return-value]
    return decorator


attempts = 0

@log_latency
@retry_on_error(max_retries=3, delay=0.01)
def call_llm_with_jitter(prompt: str) -> str:
    """模拟前两次网络抖动、第三次成功的大模型调用。"""
    global attempts
    attempts += 1
    if attempts < 3:
        raise ConnectionError("网络超时")
    return f"模型成功回复: {prompt}"

print("函数名保留验证:", call_llm_with_jitter.__name__)
print(call_llm_with_jitter("请介绍装饰器的作用"))

函数名保留验证: call_llm_with_jitter
  [Retry] 第 1 次失败: 网络超时
  [Retry] 第 2 次失败: 网络超时
[Latency] call_llm_with_jitter 耗时 20.24 ms
模型成功回复: 请介绍装饰器的作用


## 2. 第二步：生成器与 `yield` —— 惰性流式输出 `stream_tokens`
在大模型打字机流式输出（Streaming）和 RAG 长文档切分中，如果用 `return` 一次性返回一个包含几万个片段的 `list`，会瞬间吃满内存。
使用 `yield` 关键字可以把函数变成**生成器（Generator）**：每次 `next()` 只计算并吐出当前那一片，用完即释放！

In [2]:
import inspect
from collections.abc import Iterator


def stream_tokens(text: str, chunk_size: int = 2) -> Iterator[str]:
    """使用 yield 按切片流式吐出文本 Token。"""
    for idx in range(0, len(text), chunk_size):
        yield text[idx : idx + chunk_size]


print("是否为生成器函数:", inspect.isgeneratorfunction(stream_tokens))
gen = stream_tokens("智枢企业级智能体平台", chunk_size=2)
print("生成器对象:", type(gen).__name__)
print("流式切片结果:", list(gen))

是否为生成器函数: True
生成器对象: generator
流式切片结果: ['智枢', '企业', '级智', '能体', '平台']


## 3. 第三步：`asyncio` 与 `asyncio.gather` —— 批量并发请求提速
大模型 API 调用属于典型的 **I/O 密集型任务**（绝大部分时间都在等服务器返回）。
使用 `async def` 定义协程，并通过 `await asyncio.gather(*tasks)` 并发调度多个请求，可以把原本 `N * 20ms` 的串行耗时压缩到接近 `20ms`！
> 💡 在 Jupyter Notebook 中已内置事件循环，直接用 `await` 即可；在普通 `.py` 脚本中则使用 `asyncio.run(...)` 启动。

In [3]:
import asyncio


async def _async_single_ask(prompt: str, model_name: str = "deepseek-chat", delay: float = 0.02) -> str:
    await asyncio.sleep(delay)
    return f"[{model_name}] 已异步处理: {prompt}"


async def async_batch_ask(prompts: list[str], model_name: str = "deepseek-chat", delay: float = 0.02) -> list[str]:
    tasks = [_async_single_ask(p, model_name=model_name, delay=delay) for p in prompts]
    return list(await asyncio.gather(*tasks))


start = time.perf_counter()
prompts = ["什么是装饰器？", "什么是生成器？", "什么是 asyncio？", "什么是 .env？"]
batch_results = await async_batch_ask(prompts, delay=0.02)
elapsed_ms = (time.perf_counter() - start) * 1000

for res in batch_results:
    print(res)
print(f"4 条请求并发总耗时: {elapsed_ms:.2f} ms (串行约 80 ms)")

[deepseek-chat] 已异步处理: 什么是装饰器？
[deepseek-chat] 已异步处理: 什么是生成器？
[deepseek-chat] 已异步处理: 什么是 asyncio？
[deepseek-chat] 已异步处理: 什么是 .env？
4 条请求并发总耗时: 20.49 ms (串行约 80 ms)


## 4. 第四步：零依赖 `.env` 解析与密钥脱敏配置
企业研发严禁将 `API_KEY` 硬编码在源码中。我们仅用 Python 标准库 `pathlib.Path` 和 `os.environ`，即可实现安全的 `.env` 加载器，并在打印配置对象时自动脱敏密钥。

In [4]:
import os
import tempfile
from dataclasses import dataclass
from pathlib import Path


def load_env_file(env_path: str | Path, override: bool = False) -> dict[str, str]:
    parsed: dict[str, str] = {}
    for raw_line in Path(env_path).read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        key, val = line.split("=", 1)
        key, val = key.strip(), val.strip().strip('"\'')
        parsed[key] = val
        if override or key not in os.environ:
            os.environ[key] = val
    return parsed


def mask_secret(secret: str) -> str:
    return f"****...{secret[-4:]}" if len(secret) > 4 else "****"


@dataclass(frozen=True)
class AgentConfig:
    model_name: str
    api_key: str

    def __repr__(self) -> str:
        return f"AgentConfig(model_name={self.model_name!r}, api_key={mask_secret(self.api_key)!r})"


with tempfile.TemporaryDirectory() as tmpdir:
    env_file = Path(tmpdir) / ".env"
    env_file.write_text("# 演示配置\nZHISHU_MODEL_NAME=deepseek-chat\nZHISHU_API_KEY='sk-live-secret-9527'\n", encoding="utf-8")
    load_env_file(env_file, override=True)

cfg = AgentConfig(model_name=os.getenv("ZHISHU_MODEL_NAME", ""), api_key=os.getenv("ZHISHU_API_KEY", ""))
print("安全打印配置对象:", cfg)

安全打印配置对象: AgentConfig(model_name='deepseek-chat', api_key='****...9527')
